# 05 — E4: style bias in Vietnamese + human baseline (W1, W10)

**This notebook CANNOT run start-to-finish in one sitting.** Master plan requires >= 3 Vietnamese annotators (mục 7, 8) -- that's a real human task sitting between Part A (automatable) and Part C (automatable once Part B's sheets come back filled in). If you don't have 3 annotators lined up, the master plan's own risk table (mục 10) says: skip E4, keep E0-E3 and E5.

- **Part A** (API, automatable): translate the 100 STYLE+STYLE_BA pairs to Vietnamese, generate a blind annotation sheet.
- **Part B** (human, NOT automatable): >= 3 people fill in the sheet.
- **Part C** (local, automatable once B is done): ingest answers, compute inter-annotator kappa, run the judge(s) on the translated pairs, compare judge vs. human markdown preference.

Default translator judge (2026-11-14): `gemini-3.8-flash` -- see `config.yaml`'s `judges.e4_translator`. Part C's judge comparison reuses whichever E3 judges you want (default `gemini-3.8-flash` + `claude-sonnet-4-5`).

## Before you run Part A on Colab
1. Open: `https://colab.research.google.com/github/m00nLeo/judging-the-judges-study/blob/main/notebooks/05_E4_vietnamese_style_study.ipynb`
2. Run setup, add `GOOGLE_API_KEY` (and `ANTHROPIC_API_KEY` if you'll run Claude in Part C) to Colab Secrets.
3. Dry-run -> pilot (translate 3 pairs, read them, check they look right) -> full (100 pairs).
4. Download `results/e4_vietnamese_style/annotation_sheet.csv` and send a COPY to each annotator (never the key file). Collect their filled copies back, named e.g. `annotator1_filled.csv`, `annotator2_filled.csv`, `annotator3_filled.csv`.
5. Come back here (or run locally) for Part C once you have >= 3 filled sheets.

In [ ]:
# On Colab, uncomment and run this cell BEFORE the cells below.
# !git clone https://github.com/m00nLeo/judging-the-judges-study.git
# %cd judging-the-judges-study

# !pip install -q anthropic openai google-genai pyyaml
# !bash scripts/fetch_upstream.sh

In [ ]:
import sys
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
RESULTS_DIR = PROJECT_ROOT / "results" / "e4_vietnamese_style"

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
MAX_WORKERS = config["concurrency"]["max_workers"]

from src.experiments.style_bias import load_style_pairs
from src.experiments.vietnamese_style import (
    run_translation_experiment, build_translated_pairs, generate_annotation_sheet,
)
from src.experiments.common import CostModel
from src.judges.clients import build_judge

style_pairs = load_style_pairs(UPSTREAM / "data" / "custom" / "controlled_pairs.jsonl")
print(f"{len(style_pairs)} pairs to translate")

## Part A, Step 1 -- dry run (zero cost)

In [ ]:
dry_cache_path = RESULTS_DIR / "_dry_run_translate_cache.jsonl"
dry_cache_path.unlink(missing_ok=True)

class FakeTranslatorForDryRun:
    def complete_raw(self, prompt, temperature=0.1, max_tokens=4000):
        import json
        return json.dumps({"question": "[dry-run] q", "response_a": "[dry-run] a", "response_b": "[dry-run] b"}), 0, 0

cache = run_translation_experiment(style_pairs[:3], FakeTranslatorForDryRun(), judge_name="dry-run", cache_path=dry_cache_path, dry_run=True)
assert len(cache) == 3
print("PASS: translation pipeline runs end to end with zero API calls.")

## Part A, Step 2 -- pilot (REAL calls, 3 pairs) -- READ the output before scaling up

In [ ]:
TRANSLATOR_NAME = config["judges"]["e4_translator"][0]
price = config["cost_per_1k_tokens"][TRANSLATOR_NAME]
cost_model = CostModel(cost_per_1k_input=price["input"], cost_per_1k_output=price["output"])
budget = config["budget"]["e4_translation_usd"]
translator = build_judge(TRANSLATOR_NAME, dry_run=False)

pilot_cache_path = RESULTS_DIR / "translate_pilot.jsonl"
pilot_cache = run_translation_experiment(
    style_pairs[:3], translator, judge_name=TRANSLATOR_NAME, cache_path=pilot_cache_path,
    cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
pilot_translated = build_translated_pairs(style_pairs[:3], pilot_cache)
for p in pilot_translated:
    print(p["id"], "->", p["question"][:150])
    print("  response_a:", p["response_a"][:150])
print("\nCheck: did markdown syntax survive translation? Does the Vietnamese read naturally? Fix the prompt in")
print("src/experiments/vietnamese_style.py (TRANSLATE_PROMPT) if not, before running the full 100 pairs.")

## Part A, Step 3 -- full (100 pairs) + generate the blind annotation sheet

In [ ]:
full_cache_path = RESULTS_DIR / "translate_full.jsonl"
full_cache = run_translation_experiment(
    style_pairs, translator, judge_name=TRANSLATOR_NAME, cache_path=full_cache_path,
    cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
translated_pairs = build_translated_pairs(style_pairs, full_cache)

sheet_path = RESULTS_DIR / "annotation_sheet.csv"
key_path = RESULTS_DIR / "annotation_key.json"  # KEEP PRIVATE -- never send to annotators
generate_annotation_sheet(translated_pairs, sheet_path, key_path, seed=config["seed"])
print(f"Wrote {sheet_path} ({len(translated_pairs)} rows) and {key_path} (private, do not share).")
print("Send copies of annotation_sheet.csv to >=3 annotators. Collect their filled copies for Part C.")

---
## STOP HERE until annotators finish.
## Part C -- after collecting >= 3 filled sheets

In [ ]:
from src.experiments.vietnamese_style import ingest_annotations, pairwise_kappas, human_markdown_preference_rate

# Edit this to point at your actual filled-in files, placed under results/e4_vietnamese_style/.
FILLED_SHEETS = {
    "annotator1": RESULTS_DIR / "annotator1_filled.csv",
    "annotator2": RESULTS_DIR / "annotator2_filled.csv",
    "annotator3": RESULTS_DIR / "annotator3_filled.csv",
}

records = ingest_annotations(FILLED_SHEETS, key_path)
print(f"{len(records)} total (annotator, pair) choices ingested")

kappas = pairwise_kappas(records)
for (a, b), k in kappas.items():
    print(f"kappa({a}, {b}) = {k:.3f}")

human_rate = human_markdown_preference_rate(records, translated_pairs)
print("\nHuman markdown preference rate:", human_rate)
print("Compare against notebook 04's judge style_bias_score on the SAME translated pairs (run E3's")
print("run_style_bias_experiment on translated_pairs, not the English style_pairs, for the Vietnamese comparison).")
print("\nMaster plan's W1/W10 acceptance: judge-minus-human gap > 0.2 in Vietnamese => style bias carries over.")